# line profiling of optimized parallel OMP  

In [1]:
import numba
import numpy as np
from numba import jit, njit, prange

In [2]:
from OMP_numba_parallel import OMP_serial

def OMP_numba_njit_parallel_optimized(Y,T_0,D,batch_size = 1, seed=42, debug=False):
    '''
    nopython jitted OMP with parallel True and prange explicitly set. Best performance given overhead doesn't exceed sequential case.
    Improvements: removed list comprehension for norm calculations, fixing special indexing issues, attempting to rearrange axis for memory contiguity
    Future Improvements: fixing remaining memory contiguity warnings, further loop optimizationss
    '''
    # print('Hello from numba!')
    # X = np.zeros((Y.shape[0], D.shape[0])[::-1]).T # alternative to asfortranarray
    X = np.asfortranarray(np.zeros((Y.shape[0], D.shape[0])))

    # D_norm = np.linalg.norm(D, axis=1, keepdims=True)
    # D_norm = np.array([np.linalg.norm(D[i]) for i in prange(D.shape[0])])[:, np.newaxis]
    D_norm = np.sqrt(np.sum(D**2, axis=1))[:, np.newaxis]

    if debug:
        print(D.shape, D_norm.shape)
        # print(D_norm)
        
    # splits = np.arange(0, Y.shape[0], step=batch_size)
    # Y_batches = np.split(Y, splits)[1:]
    # batch_idx = np.arange(batch_size)

    n_samples = Y.shape[0]
    n_batches = (n_samples + batch_size - 1) // batch_size
    for i in prange(n_batches):
        start_idx = i * batch_size
        end_idx = min(start_idx + batch_size, n_samples)
        current_batch_size = end_idx - start_idx

        y = Y[start_idx:end_idx]

        # # ask richard what this does lol
        # if i == (len(Y_batches) - 1):
        #     batch_size = np.arange(splits[i], len(Y)).shape[0]
        #     batch_idx = batch_idx[:batch_size]
        #     if debug:
        #         print(f'batch_size = {batch_size}')

        I = np.empty((current_batch_size, T_0), dtype=np.int32)
        # D_I = np.zeros((current_batch_size, T_0, D.shape[1]))

        # Deep copy to not overwrite Y
        r = y.copy()   # (current_batch_size, D.shape[1])
        gamma = 0

        # Q = np.empty((current_batch_size, D.shape[1], T_0))
        Q = np.empty((current_batch_size, T_0, D.shape[1]))

        # Q_T_y = np.empty((current_batch_size, T_0)[::-1]).T
        Q_T_y = np.asfortranarray(np.empty((current_batch_size, T_0)))
        # R = np.empty((current_batch_size, T_0, T_0), order = 'F')
        R = np.asfortranarray(np.zeros((current_batch_size, T_0, T_0)))

        # Create a mask to ensure duplicates aren't selected
        atom_mask = np.ones((current_batch_size, D.shape[0]))

        global_dead_batches = np.zeros(current_batch_size, dtype=np.bool_)
        j_stop = T_0
        
        for j in range(T_0):
            if debug:
                print(f'Batch {i}, Iteration {j}')
                print(f'r shape: {r.shape}')

                print(f'Batch {i}, Iteration {j}')
                print(f'batch size: {current_batch_size}')



            ## got a warning for branches ending at different times
            # if np.all(global_dead_batches):
            #     j_stop = j
            #     break

            # Find best dictionary atom  

            # Let B = batch_size
            # matmul: (B, D.shape[1]) @ (D.shape[1], D.shape[0]) -> (B, D.shape[0])
            D_r = r @ D.T
            D_r *= atom_mask
            D_r = np.abs(D_r)

            # max, axis = 1: (B, D.shape[0]) -> (B,)
            # print(atom_mask.shape, batch_idx.shape)
            k = np.argmax(D_r, axis = 1)

            # print(atom_mask.shape, batch_idx.shape, k.shape)
            # atom_mask[batch_idx, k] = 0
            for b in range(current_batch_size):
                atom_mask[b, k[b]] = 0
            
            I[:, j] = k
            # D_I[:, j] = D[k]

            # Adv Indexing: (B,) -> (B, D.shape[1])
            D_k = D[k]

            if j == 0:
                # Adv Indexing: (B,) -> (B, 1)
                D_k_norm = D_norm[k]
                R[:, 0, 0] = D_k_norm[:, 0]

                # (B, dim) / (B, 1) -> (B, dim)
                # Q[:, :, 0] = D_k / D_k_norm
                for b in range(current_batch_size):
                    # Q[b, :, 0] = D_k[b] / D_k_norm[b, 0]
                    Q[b, 0] = D_k[b] / D_k_norm[b, 0]

                # for h in prange(Q.shape[0]):
                #     for l in prange(Q.shape[1]):
                #         Q[k, l, 0] = D_k[k, l] / D_k_norm[k, 0]

            else:
                if debug:
                    print(Q[:, :j].shape)
                    print(D_k[..., np.newaxis].shape)
                
                Dk_new = D_k[..., np.newaxis]
                
                # Batched mat mul: (B, 1, D.shape[1]) @ (B, D.shape[1], j) -> (B, 1, j)
                # squeeze: (B, 1, j) -> (B, j)
                # dot = np.squeeze(D_k[:, np.newaxis] @ Q[:, :, :j], axis=1)
                # R[:, 0:j, j] = dot
                # for b in range(current_batch_size):
                #     dot = Dk_new[b] @ Q[b, :, :j]
                #     R[b, 0:j, j] = dot

                # Batched mat mul: (B, j, D.shape[1]) @ (B, D.shape[1], 1) -> (B, j, 1)
                # squeeze: (B, j, 1) -> (B, j)
                # dot = np.squeeze(Q[:, :j] @ D_k[..., np.newaxis], axis=-1)
                # R[:, 0:j, j] = dot

                # print(Q[:, :j].shape, Dk_new.shape)

                dot = np.empty((current_batch_size, j))
                for b in range(current_batch_size):
                    dot[b] = (Q[b, :j] @ Dk_new[b])[:, 0]
                    # print(R[b, 0:j, j].shape, dot.shape)
                    R[b, 0:j, j] = dot[b, 0]
                
            
                # Orthogonalize
                # Batched mat mul: (B, D.shape[1], j) @ (B, j, 1) -> (B, D.shape[1], 1)
                # squeeze: (B, D.shape[1], 1) -> (B, D.shape[1])
                # subtraction: (B, D.shape[1]) - (B, D.shape[1]) -> (B, D.shape[1])

                # q_j = D_k - np.squeeze(Q[:, :, :j] @ dot[..., np.newaxis], axis = -1)
                # E_k = np.empty((current_batch_size, D.shape[1], 1))
                # q_j = np.empty((current_batch_size, D.shape[1]))
                # for b in range(current_batch_size):
                #     q_j[b] = D_k[b] - np.reshape((Q[b, :, :j] @ dot[b, :j, np.newaxis]), (1, D.shape[1]))
                # q_j = D_k - np.reshape(E_k, (current_batch_size, D.shape[1]))

                # q_j = D_k - np.squeeze(dot[:, np.newaxis] @ Q[:, :j], axis = 1)
                # E_k = np.empty((current_batch_size, D.shape[1], 1))
                q_j = np.empty((current_batch_size, D.shape[1]))
                for b in range(current_batch_size):
                    # q_j[b] = D_k[b] - np.reshape((Q[b, :, :j] @ dot[b, :j, np.newaxis]), (1, D.shape[1]))
                    # q_j[b] = D_k[b] - np.reshape((dot[b, np.newaxis, :j] @ Q[b, :j]), (1, D.shape[1]))
                    q_j[b] = D_k[b] - (dot[b, np.newaxis, :j] @ Q[b, :j])



                # q_j = D_k - np.reshape(E_k, (current_batch_size, D.shape[1]))

                
                # norm, axis = 1: (B, D.shape[1]) -> (B,)
                # q_j_norm = np.array([np.linalg.norm(q_j[l]) for l in prange(q_j.shape[0])])
                q_j_norm = np.sqrt(np.sum(q_j**2, axis=1))

                # STABILITY FIX
                # Instead of just Q[:, :, j] = q_j / q_j_norm[:, np.newaxis]
                dead_batches = (q_j_norm < 1e-15)

                global_dead_batches |= dead_batches
                q_j[dead_batches] = 0
                # atom_mask[dead_batches, k[dead_batches]] = 1
                for b in range(current_batch_size):
                    if dead_batches[b]:
                        atom_mask[b, k[b]] = 1
                
                q_j_norm_safe = q_j_norm
                q_j_norm_safe[dead_batches] = 1 # Avoid division by zero

                # Q[:, :, j] = q_j / q_j_norm_safe[:, np.newaxis]
                for b in range(current_batch_size):
                    # Q[b, :, j] = q_j[b] / q_j_norm_safe[:, np.newaxis][b, 0]
                    Q[b, j] = q_j[b] / q_j_norm_safe[:, np.newaxis][b, 0]


                
                R[:, j, j] = q_j_norm
                # Q[:, :, j] = q_j / q_j_norm[:, np.newaxis]

            # sum, axis=1: (B, D.shape[1]) -> (B,)
            # y_proj = np.sum(y * Q[:, :, j], axis = 1)
            y_proj = np.sum(y * Q[:, j], axis = 1)

            Q_T_y[:, j] = y_proj

            # mul: (B, D.shape[1]) * (B, 1) -> (B, D.shape[1])
            # r -= Q[:, :, j] * y_proj[:, np.newaxis]
            for b in range(current_batch_size):
                for f in range(D.shape[1]):
                    # r[b, f] -= Q[b, f, j] * y_proj[b]
                    r[b, f] -= Q[b, j, f] * y_proj[b]
                


        # matmul: (B, 1, D.shape[1]) @ (B, D.shape[1], j_stop) -> (B, 1, j_stop) 
        # transpose (0, 2, 1): (B, 1, j_stop) -> (B, j_stop, 1)
        # Q_T_y = np.transpose(y[:, np.newaxis] @ Q[:, :, :j_stop+1], (0, 2, 1))
        # gamma = np.linalg.solve(R[:, :j_stop, :j_stop], Q_T_y[:, :j_stop, np.newaxis])
        # gamma = np.squeeze(gamma, axis=-1)

        gamma = np.zeros((current_batch_size, j_stop))

        for b in range(current_batch_size):
            gamma[b, :] = np.linalg.solve(
                R[b, :j_stop, :j_stop],
                Q_T_y[b, :j_stop]
            )

        '''
        Chat says this is the best soln for some reason, need to validate.
        gamma = np.zeros((current_batch_size, j_stop))

        for b in range(current_batch_size):
            for i in range(j_stop - 1, -1, -1):  # backward
                s = Q_T_y[b, i]
                
                for k in range(i + 1, j_stop):
                    s -= R[b, i, k] * gamma[b, k]
                
                gamma[b, i] = s / R[b, i, i]
        '''
        
        # rows = 0
        # cols = I[:, :j_stop]
        # if start_idx + batch_size >= n_samples:
        #     if debug:
        #         # print(splits[i].dtype)
        #         # print(I.dtype)
        #         print(f'gamma: {gamma}')
        #         print(f'gamma shape: {gamma.shape}')
        #         print(f'k shape: {k.shape}')
        #         print(f'I shape: {I.shape}')
        #     rows = np.arange(splits[i], len(Y))[:, np.newaxis]
            
        # elif i == 0:
        #     rows = np.arange(0, splits[1])[:, np.newaxis]
        # else:
        #     rows = np.arange(splits[i], splits[i+1])[:, np.newaxis]

        # # X[rows, cols] = gamma
        # if debug:
        #     print(f'rows: ', rows)
        #     print(f'cols: ', cols)
        #     print(f'X shape is: ', X.shape)
        #     print(f'gamma shape is: ', gamma.shape)
        # for l in prange(rows.shape[0]):        # batch dimension
        #     row = rows[l, 0]
        #     for h in prange(cols.shape[1]):   # atoms per signal
        #         col = cols[l, h]
        #         X[row, col] = gamma[l, h]
        
        for b in range(current_batch_size):
            row = start_idx + b
            for h in range(j_stop):
                col = I[b, h]
                X[row, col] = gamma[b, h]
            
        if debug:
            print('success!')
    return X

In [3]:
from sklearn.datasets import make_regression

T0 = 3
batch_size = 2
D, y = make_regression(n_samples = 50, n_features = 300, n_targets = 20_000, noise=4, random_state=0)
# print(D.shape, y.shape)

X1 = OMP_serial(y.T, T0, D.T, batch_size=batch_size, debug=False)

X2 = OMP_numba_njit_parallel_optimized(y.T, T0, D.T, batch_size=batch_size, debug=False)

# compare the loss between X1 and X2
loss1 = np.linalg.norm(y.T - X1 @ D.T)
loss2 = np.linalg.norm(y.T - X2 @ D.T)
print(f'Loss of OMP_serial: {loss1}')
print(f'Loss of OMP_numba_njit_parallel_optimized: {loss2}')

Loss of OMP_serial: 100857.26490637184
Loss of OMP_numba_njit_parallel_optimized: 101970.42506323293


In [4]:
%load_ext line_profiler

In [5]:
from line_profiler import profile

In [6]:
l_OMP_numba_njit_parallel_optimized = profile(OMP_numba_njit_parallel_optimized)
%lprun -f l_OMP_numba_njit_parallel_optimized l_OMP_numba_njit_parallel_optimized(Y = y.T, \
               T_0 = T0, \
                D = D.T, \
                batch_size = batch_size, \
                debug=False) 

Timer unit: 1e-09 s

Total time: 6.97472 s
File: /tmp/ipykernel_706654/2952063050.py
Function: OMP_numba_njit_parallel_optimized at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def OMP_numba_njit_parallel_optimized(Y,T_0,D,batch_size = 1, seed=42, debug=False):
     4                                               '''
     5                                               nopython jitted OMP with parallel True and prange explicitly set. Best performance given overhead doesn't exceed sequential case.
     6                                               Improvements: removed list comprehension for norm calculations, fixing special indexing issues, attempting to rearrange axis for memory contiguity
     7                                               Future Improvements: fixing remaining memory contiguity warnings, further loop optimizationss
     8                                               '''
     9             